# IPC2BNS-Verify — Phase 6: Master Evaluation, Ablation Summary & Final Report

This master notebook runs the complete end-to-end evaluation harness across all 4 stages:
1. **Stage 1 (Baseline LLM, Closed-Book)**
2. **Stage 2 (+RAG Statutory Context)**
3. **Stage 3 (+Two-Layer Hard-Constraint Verifier)**
4. **Stage 4 (+Verifier + Incremental Refresh)**
5. **Master Ablation Summary Table Generation** (`ablation_summary_table.csv`)
6. **Full 65-Test Automated Pytest Suite Execution**

---
## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os, sys, shutil

# Mount Drive cleanly
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = '/content/drive/MyDrive/NLP_rspaper'
LOCAL_ROOT = '/content/IPC2BNS-Verify'

# Copy to Colab local SSD for lightning-fast disk I/O & zero network timeouts
if os.path.exists(DRIVE_ROOT):
    if os.path.exists(LOCAL_ROOT):
        shutil.rmtree(LOCAL_ROOT)
    shutil.copytree(DRIVE_ROOT, LOCAL_ROOT, ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache', '.git'))
    PROJECT_ROOT = LOCAL_ROOT
    print('✅ Synced project from Drive to Colab local SSD:', PROJECT_ROOT)
else:
    PROJECT_ROOT = DRIVE_ROOT

os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT
code_dir = os.path.join(PROJECT_ROOT, 'code')
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

print('Environment initialized.')


---
## 2. Dependencies

In [ ]:
!pip install -q pytest pandas tabulate
print('Dependencies ready.')

---
## 3. Run Master Evaluation Harness Across All 4 Stages

In [ ]:
import os, sys, importlib
importlib.invalidate_caches()

# Ensure code directory is at top of sys.path
code_dir = os.path.join(PROJECT_ROOT, 'code')
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

# Force fresh reload if already loaded in this runtime session
for mod in ['src.eval.harness', 'src.eval', 'src']:
    if mod in sys.modules:
        try:
            importlib.reload(sys.modules[mod])
        except Exception:
            pass

from src.eval.harness import MasterEvaluationHarness, generate_full_ablation_report

results_dir = os.path.join(PROJECT_ROOT, 'results')
out_csv = os.path.join(results_dir, 'ablation_summary_table.csv')

harness = MasterEvaluationHarness(results_dir)
ablation_rows = harness.export_ablation_summary_csv(out_csv)

import pandas as pd
df = pd.DataFrame(ablation_rows)
print('\n' + '='*85)
print('MASTER ABLATION SUMMARY TABLE')
print('='*85)
display(df)


---
## 4. Human Expert Calibration Inspection

In [ ]:
import os
import pandas as pd

human_cal_file = os.path.join(PROJECT_ROOT, 'results/human_review_calibration.csv')
cal_df = pd.read_csv(human_cal_file)
print('=== DOUBLE-BLIND LEGAL EXPERT CALIBRATION (SAMPLE) ===')
# Select relevant calibration columns present in the dataset
desired_cols = ['question_id', 'legal_expert_1_score', 'legal_expert_1_verdict', 'legal_expert_2_score', 'legal_expert_2_verdict', 'consensus_verdict', 'verifier_alignment_status']
display_cols = [c for c in desired_cols if c in cal_df.columns]
display(cal_df[display_cols].head(10))


---
## 5. Qualitative Error Analysis Notes

In [ ]:
error_notes_file = os.path.join(PROJECT_ROOT, 'results/error_analysis_notes.md')
with open(error_notes_file, 'r', encoding='utf-8') as f:
    print(f.read())

---
## 6. Run Complete Automated Pytest Suite (All 65 Tests)

In [ ]:
import os
test_dir = os.path.join(PROJECT_ROOT, 'code/tests')
code_dir = os.path.join(PROJECT_ROOT, 'code')
# Run pytest with code directory in PYTHONPATH
!PYTHONPATH="{code_dir}" python -m pytest "{test_dir}" -v --color=yes


---
## 7. Check Final WBS Project Completion

In [ ]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report

---
## 8. Sync Results Back to Google Drive

In [ ]:
if PROJECT_ROOT == LOCAL_ROOT:
    import shutil, os
    os.makedirs(os.path.join(DRIVE_ROOT, 'results'), exist_ok=True)
    for f in ['ablation_summary_table.csv', 'progress_report.md']:
        src_f = os.path.join(PROJECT_ROOT, 'results', f)
        if os.path.exists(src_f):
            shutil.copy2(src_f, os.path.join(DRIVE_ROOT, 'results', f))
    print('✅ Saved latest results to Google Drive successfully.')
